## **Learning Objectives**

By completing these exercises, you will:

- Understand Retrieval-Augmented Generation (RAG) and its components.
- Load, preprocess, and handle PDF documents effectively.
- Convert textual data into embeddings for efficient retrieval.
- Implement and test document retrieval systems using LangChain and FAISS.
- Integrate retrieval systems with free Language Models (LLMs) from ChatGroq .
- Build an interactive chat-based Q&A system.

---

## **Exercise 1: Setup and Warm-up**

In this exercise, you'll set up your environment and select a suitable language model.

**Steps:**

1. **Load Environment Variables:** Ensure your environment variables (e.g., API keys, tokens) are securely stored and loaded.
2. **Choose LLM:** Select a free LLM model from from ChatGroq. 
3. **Instantiate the Model:** Create an instance of your chosen model.


In [1]:
# Import necessary libraries
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint

# Load environment variables
load_dotenv()



c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-rag-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [22]:
# Import necessary libraries
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# Instantiate the UPDATED Groq model
llm = ChatGroq(
    model="llama-3.1-8b-instant", # <-- THIS IS THE NEW FIX
    temperature=0.1, 
    max_tokens=512
)
print("Groq LLM instantiated successfully with Llama 3.1!")

Groq LLM instantiated successfully with Llama 3.1!


---

## **Exercise 2: Data Ingestion**

In this exercise, you'll learn to load PDF data into a Python environment.

**Steps:**

1. **Import PDF Loader:** Use LangChain’s `PyPDFLoader`.
2. **Load PDF File:** Create a function to read the PDF file.
3. **Display PDF Content:** Print the number of pages and first page content.

In [ ]:
# Import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# Example function to load PDF

def load_pdf(pdf_path):
    pass  # Implement PDF loading here

In [7]:
# Import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# 1. Keep the function generic with a placeholder variable (pdf_path)
def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# 2. Pass your actual file path down here when you call the function!
# Notice the 'r' before the quotes to safely handle Windows backslashes
my_file_path = r"C:\Users\thngu\neuefische_course\pepperprompt\fork-ds-rag-pipeline\documents\UNU-INWEH-Report-The_Env_Cost_of_AI-2026.pdf"

# 3. Test it
docs = load_pdf(my_file_path)
print(f"Loaded {len(docs)} pages.")
print("="*50)
print(docs[0].page_content[:200]) # Print first 200 characters of page 1

Loaded 56 pages.
ENVIRONMENTAL 
COST OF AI'S 
ENERGY USE
United Nations University
Institute for Water, Environment and Health
Carbon, Water and Land Footprints


---

## **Exercise 3: Document Chunking**

This exercise introduces splitting large documents into manageable text chunks.

**Steps:**

1. **Import Text Splitter:** Use `RecursiveCharacterTextSplitter`.
2. **Chunk Document:** Write a function that splits loaded documents into chunks.
3. **Test Function:** Verify by displaying the resulting chunks.


In [8]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Example chunking function
def chunk_documents(documents, chunk_size=200, chunk_overlap=50):
    pass  # Implement your chunking logic

In [14]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(documents, chunk_size=800, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    return chunks

# Test it
doc_chunks = chunk_documents(docs)
print(f"Split document into {len(doc_chunks)} chunks.")

Split document into 221 chunks.



---

## **Exercise 4: Embedding and Storage**

In this exercise, you will create embeddings from text chunks and store them efficiently.

**Steps:**

1. **Choose Embedding Model:** Use `sentence-transformers/all-mpnet-base-v2` from Hugging Face.
2. **Generate Embeddings:** Transform document chunks into embeddings.
3. **Store Embeddings:** Save these embeddings using FAISS locally.


In [ ]:
# Import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Example function for embeddings and storage
def embed_and_store(chunks):
    pass  # Implement your embedding creation and storage logic

In [15]:
# Generate embeddings and save them locally

# Import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

def embed_and_store(chunks):
    # Create the FAISS index in memory
    vectorstore = FAISS.from_documents(chunks, embeddings_model)
    # Save it to a local folder named "faiss_index"
    vectorstore.save_local("faiss_index")
    print("Embeddings successfully saved to local FAISS index!")
    return vectorstore

# Run it
vector_db = embed_and_store(doc_chunks)

Embeddings successfully saved to local FAISS index!


---

## **Exercise 5: Retrieval from FAISS**

Here, you will learn how to retrieve documents from a vector database using embeddings.

**Steps:**

1. **Load Embeddings:** Load stored embeddings from the FAISS database.
2. **Implement Retrieval:** Create logic to retrieve relevant chunks based on queries.
3. **Test Retriever:** Execute retrieval using sample queries.

In [17]:
# Implement retrieval logic from your FAISS database

# Load the stored embeddings
vectorstore = FAISS.load_local(
    "faiss_index", 
    embeddings_model, 
    allow_dangerous_deserialization=True # Required by recent LangChain updates
)

# Create the retriever (fetching the top 3 most relevant chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test the search engine (No LLM yet, just pure search)
query = "What is the water footprint of training Large Language Models?"
search_results = retriever.invoke(query)

print(f"Top 3 snippets retrieved for: '{query}'")
print("="*80)
for i, chunk in enumerate(search_results):
    # We use .get() here in case the PDF loader didn't grab the page number cleanly
    page_num = chunk.metadata.get('page', 'Unknown Page')
    print(f"\n--- Snippet {i+1} (From page {page_num}) ---")
    print(chunk.page_content)

Top 3 snippets retrieved for: 'What is the water footprint of training Large Language Models?'

--- Snippet 1 (From page 39) ---
Environmental Cost of AI's Energy Use
40
forms of AI, but they do not draw the same 
power. Text classification typically uses orders 
of magnitude less energy per query than 
image generation or long-form text synthesis. 
And even within a single task, complexity 
varies: model size, context window, prompt 
length, output length, and other settings 
can substantially shift the per-query energy 
use. Model choice matters as each AI model 
performs each task with a different energy and 
environmental cost. Even when using the same 
AI platform, users' choices of which model 
to run (e.g., Instant vs. Thinking) can change 
the energy use. Model choice also makes the 
location of AI operations relevant, because the 
same watt-hour carries different carbon, water,

--- Snippet 2 (From page 38) ---
each interaction draws electricity, multiplied 
billions of times,

In [ ]:
# Test your retrieval system with queries

## Comment:

This is a textbook RAG retrieval! Your local vector database successfully searched all 56 pages and instantly returned the top 3 most mathematically relevant paragraphs, complete with page metadata.

However, as an AI Project Manager, I look at this output and see a massive Quality Assurance (QA) lesson staring right at us.
The PM Analysis: The "Needle" vs. The "Concept"

Look closely at the text it retrieved. Your question was: "What is the water footprint of training Large Language Models?"

    Snippet 3 tells us why there is a water footprint: "a water footprint from electricity production and cooling."

    Snippet 1 & 2 tell us how different tasks and models affect operational energy.

    What is missing? The actual, quantitative number (e.g., "Training GPT-3 consumed 700,000 liters of water").

This happens often in RAG. The vector database found the concept perfectly, but the top 3 chunks happened to not contain the specific mathematical statistic.

This leads us to the ultimate test of your roadmap's Trust & Accuracy goal: How will the LLM handle this? Will it hallucinate a number to please you, or will it follow our strict prompt instructions and admit it doesn't have the exact number based on these three chunks?

---

## **Exercise 6: Connecting Retrieval with LLM**

You'll now connect document retrieval with the Language Model.

**Steps:**

1. **Create Retrieval Chain:** Link your retrieval system to your instantiated LLM.
2. **Test the Chain:** Confirm it works by generating answers from retrieved documents.

In [23]:
# Import the required LangChain modules
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 1. THE BLUEPRINT: Define the function (This was the missing piece!)
def create_rag_chain(llm, retriever):
    system_prompt = (
        "You are an expert AI data analyst. "
        "Use the following pieces of retrieved context to answer the question. "
        "If the specific answer (like an exact number) is not in the context, explicitly say "
        "'I don't know based on the provided documents' before providing any general context.\n\n"
        "Context:\n{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    stuff_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, stuff_chain)
    
    return rag_chain

# 2. EXECUTION: Create the complete RAG chain
qa_chain = create_rag_chain(llm, retriever)

# 3. QA TEST: Pass the exact same query through the full pipeline
query = "What is the water footprint of training Large Language Models?"
print(f"Sending query to Groq/Llama 3: '{query}'")
print("="*80)

# Invoke the chain
response = qa_chain.invoke({"input": query})

print("\n🤖 AI Final Answer:")
print(response["answer"])
print("\n" + "="*80)
print("Was this answer grounded in the 3 chunks we saw earlier?")

Sending query to Groq/Llama 3: 'What is the water footprint of training Large Language Models?'

🤖 AI Final Answer:
I don't know based on the provided documents. The context mentions that training concentrates electricity use over weeks or months, and that every kilowatt-hour of electricity used to train or run an AI model carries environmental footprints, including a water footprint from electricity production and cooling. However, it does not provide specific information about the water footprint of training Large Language Models.

Was this answer grounded in the 3 chunks we saw earlier?


### Expected Answer: 


Sending query to Groq/Llama 3: 'What is the water footprint of training Large Language Models?'

================================================================================

🤖 AI Final Answer:
I don't know based on the provided documents. The context mentions that training concentrates electricity use over weeks or months, and that every kilowatt-hour of electricity used to train or run an AI model carries environmental footprints, including a water footprint from electricity production and cooling. However, it does not provide specific information about the water footprint of training Large Language Models.

================================================================================

Was this answer grounded in the 3 chunks we saw earlier?

================================================================================

YES! 100% YES! As your AI Project Manager, I am giving this output a standing ovation. This is a perfect, textbook pass for our Trust & Accuracy product goal.

Here is exactly why this output is a massive victory for your RAG MVP:
The PM Breakdown: Why This is Perfect

    It Followed the System Prompt: We explicitly told it in the prompt: "If the specific answer (like an exact number) is not in the context, explicitly say 'I don't know based on the provided documents'." It followed that instruction to the letter.

    Zero Hallucination: If you asked ChatGPT this same question without RAG, it might have scraped the internet and confidently told you that training GPT-3 took 700,000 liters of water. But because we locked it into our specific UN Report, it refused to guess.

    Perfect Synthesis: Even though it didn't have the exact number, it perfectly read the 3 chunks and summarized the cause of the footprint (electricity production and cooling) without making anything up.

If a customer support agent uses this tool, they can trust it implicitly. It will never give a customer a fake refund policy just to sound helpful.
You Are Ready for Launch! 🚀

Your Data Capex is built, your FinOps chunking strategy is deployed, and your Quality Assurance test just passed with flying colors.

Go ahead and run that final Exercise 7 cell:

In [24]:
# Invoke your chain with a sample question

---

## **Exercise 7: Interactive Chat System**

In the final exercise, build an interactive chat-based query system.

**Steps:**

1. **Create Chat Interface:** Develop a simple function for interactive querying.
2. **Run the Chat:** Allow users to ask questions and receive immediate responses.


In [27]:
# Define your interactive chat querying function

def interactive_chat(chain):
    print("="*50)
    print("🤖 Welcome to the AI Support Bot!")
    print("Type 'exit' or 'quit' to end the session.")
    print("="*50)
    
    while True:
        user_input = input("\nYou: ")
        
        # Check for exit command
        if user_input.lower() in ['exit', 'quit']:
            print("AI: Goodbye! Have a great day.")
            break
            
        # Get response from the RAG chain
        try:
            response = chain.invoke({"input": user_input})
            print(f"AI: {response['answer']}")
        except Exception as e:
            print(f"AI: Oops, something went wrong: {e}")



In [ ]:
# Run and test your interactive chat system

# Run the chat interface!
interactive_chat(qa_chain)

---

## **Conclusion & Reflection**

After completing these exercises:

- Summarize key concepts learned.
- Reflect on the effectiveness and limitations of the free LLM and RAG system you've built.
- Consider how you might improve or extend your system in practical applications.

---